# Regression-based physical copper price and certificate bubble

This notebook is an experimental alternative to ratio interpolation. It estimates IME physical copper price from free-market USD/IRR and LME cash copper USD/kg, compares a linear model with a degree-2 polynomial Ridge model using time-series cross-validation, and calculates the certificate bubble from the selected model. The production pipeline is not modified.

In [27]:
from bisect import bisect_right
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 1. Load the four source datasets

In [28]:
def first_existing(*candidates):
    path = next((Path(candidate) for candidate in candidates if Path(candidate).exists()), None)
    if path is None:
        raise FileNotFoundError(f"None of these paths exists: {candidates}")
    return path


certificate_path = first_existing(
    "../data/raw/certificate/copper_certificate_raw.csv",
    "commodity/copper/data/raw/certificate/copper_certificate_raw.csv",
)
physical_path = first_existing(
    "../data/processed/physical/nci_copper_cash_daily.csv",
    "commodity/copper/data/processed/physical/nci_copper_cash_daily.csv",
)
lme_path = first_existing(
    "../data/raw/lme/copper_lme_raw.csv",
    "commodity/copper/data/raw/lme/copper_lme_raw.csv",
)
usd_path = first_existing(
    "../../../shared/data/raw/fx/usd_to_rial.csv",
    "shared/data/raw/fx/usd_to_rial.csv",
)

certificate_raw = pd.read_csv(certificate_path)
physical_daily = pd.read_csv(physical_path)
lme_raw = pd.read_csv(lme_path)
usd_raw = pd.read_csv(usd_path)

{
    "certificate_raw": certificate_raw.shape,
    "physical_daily": physical_daily.shape,
    "lme_raw": lme_raw.shape,
    "usd_raw": usd_raw.shape,
}

{'certificate_raw': (288, 32),
 'physical_daily': (799, 16),
 'lme_raw': (4733, 7),
 'usd_raw': (13098, 5)}

## 2. Clean dates and prices

Certificate price is daily VWAP (`TradesValue / TradesVolume`). LME is converted from USD/metric-tonne to USD/kg. Missing weekend or holiday LME/USD observations are joined backward, never from a future date.

In [29]:
certificate = certificate_raw.copy()
certificate["date"] = pd.to_datetime(certificate["DT"].str[:10])
for column in ["TradesVolume", "TradesValue", "TodaySettlementPrice"]:
    certificate[column] = pd.to_numeric(certificate[column], errors="raise")
certificate = certificate.loc[certificate["TradesVolume"] > 0].copy()
certificate["certificate_price"] = (
    certificate["TradesValue"] / certificate["TradesVolume"]
)
certificate["settlement_check_error"] = (
    certificate["certificate_price"] - certificate["TodaySettlementPrice"]
).abs()
assert certificate["settlement_check_error"].max() <= 0.500001
certificate = certificate[[
    "date", "certificate_price", "TradesVolume", "TradesValue"
]].sort_values("date")

physical = physical_daily.copy()
physical["date"] = pd.to_datetime(physical["physical_trade_date_gregorian"])
physical["physical_price"] = pd.to_numeric(
    physical["physical_weighted_price"], errors="raise"
)
physical = physical[["date", "physical_price", "total_quantity", "physical_trades_value_irr"]].sort_values("date")

lme = lme_raw.loc[lme_raw["cash_settlement"].astype(str).str.strip().ne("-")].copy()
lme["lme_source_date"] = pd.to_datetime(lme["date"])
lme["lme_usd_per_ton"] = pd.to_numeric(
    lme["cash_settlement"].astype(str).str.replace(",", "", regex=False),
    errors="raise",
)
lme["lme_usd_per_kg"] = lme["lme_usd_per_ton"] / 1_000
lme = lme[["lme_source_date", "lme_usd_per_kg"]].sort_values("lme_source_date")

def parse_mixed_gregorian(value):
    parts = [int(part) for part in str(value).replace("-", "/").split("/")]
    if parts[0] >= 1900:
        year, month, day = parts
    else:
        month, day, year = parts
    return pd.Timestamp(year=year, month=month, day=day)

usd = usd_raw.copy()
usd["usd_source_date"] = usd["date_gr"].map(parse_mixed_gregorian)
usd["usd_irr"] = pd.to_numeric(
    usd["price_irr"].astype(str).str.replace(",", "", regex=False),
    errors="raise",
)
usd = usd[["usd_source_date", "usd_irr"]].sort_values("usd_source_date")

assert certificate["date"].is_unique
assert physical["date"].is_unique
assert lme["lme_source_date"].is_unique
assert usd["usd_source_date"].is_unique

## 3. Add backward-looking LME and USD inputs

In [30]:
def add_market_inputs(frame):
    result = frame.sort_values("date").copy()
    result = pd.merge_asof(
        result,
        lme,
        left_on="date",
        right_on="lme_source_date",
        direction="backward",
    )
    result = pd.merge_asof(
        result.sort_values("date"),
        usd,
        left_on="date",
        right_on="usd_source_date",
        direction="backward",
    )
    result["lme_age_days"] = (result["date"] - result["lme_source_date"]).dt.days
    result["usd_age_days"] = (result["date"] - result["usd_source_date"]).dt.days
    result["intrinsic_price"] = result["lme_usd_per_kg"] * result["usd_irr"]
    if result[["lme_usd_per_kg", "usd_irr"]].isna().any().any():
        raise ValueError("Missing LME or USD inputs after backward merge")
    return result


# Regression training data comes only from the physical market, USD, and LME.
# The certificate dataset is not merged into the regression sample.
REGRESSION_START_DATE = pd.Timestamp("2025-11-12")
physical_regression_sample = physical.loc[
    physical["date"] >= REGRESSION_START_DATE
].copy()
anchors = add_market_inputs(physical_regression_sample)
assert not anchors.empty, "No physical observations found in the regression period"

# Certificate market inputs are prepared separately and are used only after fitting.
certificate_features = add_market_inputs(certificate)

anchors[[
    "date", "physical_price", "usd_irr", "lme_usd_per_kg", "intrinsic_price"
]]

,date,physical_price,usd_irr,lme_usd_per_kg,intrinsic_price
0,2025-11-23,"10,660,673.0000","1,132,000.0000",10.6855,"12,095,986.0000"
1,2025-12-17,"13,266,993.0000","1,310,800.0000",11.7200,"15,362,576.0000"
2,2026-03-29,"16,541,000.0000","1,585,600.0000",12.0460,"19,100,137.6000"
3,2026-03-30,"16,541,000.0000","1,585,600.0000",12.1370,"19,244,427.2000"
4,2026-04-05,"16,541,000.0000","1,527,800.0000",12.1470,"18,558,186.6000"
5,2026-04-12,"16,766,000.0000","1,553,100.0000",12.6605,"19,663,022.5500"
6,2026-04-13,"16,766,000.0000","1,578,300.0000",12.8205,"20,234,595.1500"
7,2026-04-19,"17,604,000.0000","1,507,300.0000",13.1490,"19,819,487.7000"
8,2026-04-26,"18,484,000.0000","1,569,500.0000",13.2300,"20,764,485.0000"
9,2026-05-03,"19,308,847.0000","1,869,000.0000",12.8950,"24,100,755.0000"


## 4. Compare linear and degree-2 polynomial regression

The regression uses the currently available physical observations from the stated start date. The polynomial model is regularized with Ridge. `TimeSeriesSplit` preserves temporal order, and selection uses the lowest out-of-sample RMSE. Certificate prices are not present in `X`, `y`, or the regression sample.

In [31]:
FEATURES = ["usd_irr", "lme_usd_per_kg"]
TARGET = "physical_price"

X = anchors[FEATURES]
y = anchors[TARGET]

models = {
    "linear": Pipeline([
        ("scale", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "polynomial_degree_2_ridge": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-4, 4, 81))),
    ]),
}

tscv = TimeSeriesSplit(n_splits=5)
comparison_rows = []
oos_predictions = {}

for name, model in models.items():
    prediction = pd.Series(np.nan, index=y.index, dtype=float)
    for train_index, test_index in tscv.split(X):
        fold_model = clone(model)
        fold_model.fit(X.iloc[train_index], y.iloc[train_index])
        prediction.iloc[test_index] = fold_model.predict(X.iloc[test_index])
    valid = prediction.notna()
    valid_prediction = prediction.loc[valid]
    actual = y.loc[valid]
    oos_predictions[name] = valid_prediction
    comparison_rows.append({
        "model": name,
        "oos_n": len(actual),
        "MAE": mean_absolute_error(actual, valid_prediction),
        "RMSE": mean_squared_error(actual, valid_prediction) ** 0.5,
        "R2": r2_score(actual, valid_prediction),
    })

model_comparison = pd.DataFrame(comparison_rows).sort_values("RMSE").reset_index(drop=True)
model_comparison

,model,oos_n,MAE,RMSE,R2
0,linear,25,"448,354.0371","620,006.1661",0.9087
1,polynomial_degree_2_ridge,25,"632,242.2276","1,011,988.8939",0.7568


## 5. Fit the selected model and inspect anchor fit

In [32]:
selected_model_name = model_comparison.loc[0, "model"]
selected_model = models[selected_model_name]
selected_model.fit(X, y)

anchors["fitted_physical_price"] = selected_model.predict(X)
anchors["fitted_error_pct"] = (
    anchors["fitted_physical_price"] / anchors["physical_price"] - 1
) * 100

print(f"Selected model: {selected_model_name}")
if selected_model_name == "polynomial_degree_2_ridge":
    print(f"Selected Ridge alpha: {selected_model.named_steps['model'].alpha_}")

anchors[[
    "date", "physical_price", "fitted_physical_price", "fitted_error_pct"
]]

Selected model: linear


,date,physical_price,fitted_physical_price,fitted_error_pct
0,2025-11-23,"10,660,673.0000","10,732,909.0703",0.6776
1,2025-12-17,"13,266,993.0000","13,696,038.2544",3.2339
2,2026-03-29,"16,541,000.0000","16,350,557.8007",-1.1513
3,2026-03-30,"16,541,000.0000","16,487,320.3835",-0.3245
4,2026-04-05,"16,541,000.0000","16,047,062.9999",-2.9861
5,2026-04-12,"16,766,000.0000","17,018,080.9079",1.5035
6,2026-04-13,"16,766,000.0000","17,457,041.0904",4.1217
7,2026-04-19,"17,604,000.0000","17,391,477.3425",-1.2072
8,2026-04-26,"18,484,000.0000","18,003,155.7767",-2.6014
9,2026-05-03,"19,308,847.0000","19,858,828.0275",2.8483


In [33]:
limits = [min(anchors['physical_price'].min(), anchors['fitted_physical_price'].min()),
          max(anchors['physical_price'].max(), anchors['fitted_physical_price'].max())]
fig = go.Figure()
fig.add_trace(go.Scatter(x=anchors['physical_price'], y=anchors['fitted_physical_price'],
    mode='markers', name='Physical anchors', marker=dict(color='#1976D2', size=8)))
fig.add_trace(go.Scatter(x=limits, y=limits, name='Perfect fit',
    line=dict(color='#455A64', dash='dash')))
fig.update_layout(title=f'Observed vs fitted IME physical price: {selected_model_name}',
    xaxis_title='Observed IRR/kg', yaxis_title='Fitted IRR/kg',
    height=550, template='plotly_white')
fig.show()

## 6. Estimate physical price for all available certificate trading days and calculate bubble

Unlike linear ratio interpolation, regression can produce estimates before the first and after the last physical anchor. These are model estimates—not observed physical trades—and should be interpreted with the cross-validation results.

In [34]:
bubble_regression = certificate_features.copy()
bubble_regression["estimated_physical_price"] = selected_model.predict(
    bubble_regression[FEATURES]
)
if (bubble_regression["estimated_physical_price"] <= 0).any():
    raise ValueError("The selected regression produced a non-positive physical price")

bubble_regression["certificate_bubble_irr_per_kg"] = (
    bubble_regression["certificate_price"]
    - bubble_regression["estimated_physical_price"]
)
bubble_regression["certificate_bubble_pct"] = (
    bubble_regression["certificate_price"]
    / bubble_regression["estimated_physical_price"]
    - 1
) * 100
bubble_regression["is_physical_anchor_date"] = bubble_regression["date"].isin(anchors["date"])
bubble_regression["regression_model"] = selected_model_name

bubble_regression[[
    "date", "certificate_price", "estimated_physical_price",
    "certificate_bubble_pct", "is_physical_anchor_date"
]].describe(include="all")

,date,certificate_price,estimated_physical_price,certificate_bubble_pct,is_physical_anchor_date
count,210,210.0000,210.0000,210.0000,210
unique,NaN,NaN,NaN,NaN,2
top,NaN,NaN,NaN,NaN,False
freq,NaN,NaN,NaN,NaN,175
mean,2026-04-05 06:58:17.142857,"18,815,057.7009","17,794,889.0775",4.9615,NaN
min,2025-10-20 00:00:00,"10,205,657.0950","10,163,304.5834",-14.6249,NaN
25%,2026-01-04 06:00:00,"15,649,877.1917","15,628,783.5066",-0.6557,NaN
50%,2026-04-14 00:00:00,"18,166,352.9640","17,890,374.7827",5.1122,NaN
75%,2026-06-29 18:00:00,"23,034,847.5422","20,380,811.4282",10.9266,NaN
max,2026-09-20 00:00:00,"29,384,188.9552","26,137,087.0469",24.0637,NaN


In [35]:
anchor_bubbles = bubble_regression.loc[bubble_regression['is_physical_anchor_date']]
fig = go.Figure()
fig.add_trace(go.Scatter(x=bubble_regression['date'], y=bubble_regression['certificate_bubble_pct'],
    name=f'Regression bubble ({selected_model_name})', line_color='#1976D2'))
fig.add_trace(go.Scatter(x=anchor_bubbles['date'], y=anchor_bubbles['certificate_bubble_pct'],
    mode='markers', name='Physical anchor date', marker=dict(color='#C62828', size=8)))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title='Daily copper certificate bubble: regression estimate',
    xaxis_title='Date', yaxis_title='Bubble (%)', height=550,
    template='plotly_white', hovermode='x unified')
fig.show()

## 7. Optional: inspect or save the experimental result

The save command is intentionally commented out. Uncomment it only if the regression result is accepted after reviewing cross-validation diagnostics.

In [36]:
regression_output = bubble_regression[[
    "date",
    "certificate_price",
    "TradesVolume",
    "lme_source_date",
    "lme_age_days",
    "lme_usd_per_kg",
    "usd_source_date",
    "usd_age_days",
    "usd_irr",
    "intrinsic_price",
    "estimated_physical_price",
    "certificate_bubble_irr_per_kg",
    "certificate_bubble_pct",
    "is_physical_anchor_date",
    "regression_model",
]].copy()

regression_output.head()

# Optional save after reviewing model diagnostics:
# output_path = Path("data/processed/bubble/copper_certificate_bubble_regression.csv")
# output_path.parent.mkdir(parents=True, exist_ok=True)
# regression_output.to_csv(output_path, index=False, encoding="utf-8-sig")

,date,certificate_price,TradesVolume,lme_source_date,lme_age_days,lme_usd_per_kg,usd_source_date,usd_age_days,usd_irr,intrinsic_price,estimated_physical_price,certificate_bubble_irr_per_kg,certificate_bubble_pct,is_physical_anchor_date,regression_model
0,2025-10-20,"10,507,161.1641",57651,2025-10-20,0,10.5810,2025-10-20,0,"1,081,500.0000","11,443,351.5000","10,178,072.8434","329,088.3206",3.2333,False,linear
1,2025-10-21,"10,454,394.0267",26322,2025-10-21,0,10.6120,2025-10-21,0,"1,075,500.0000","11,413,206.0000","10,177,400.7474","276,993.2793",2.7217,False,linear
2,2025-10-22,"10,215,889.4684",90627,2025-10-22,0,10.6000,2025-10-22,0,"1,076,000.0000","11,405,600.0000","10,163,304.5834","52,584.8850",0.5174,False,linear
3,2025-10-25,"10,205,657.0950",13043,2025-10-24,1,10.8070,2025-10-25,0,"1,072,000.0000","11,585,104.0000","10,442,894.1888","-237,237.0938",-2.2718,False,linear
4,2025-10-26,"10,335,702.5324",70918,2025-10-24,2,10.8070,2025-10-26,0,"1,078,000.0000","11,649,946.0000","10,490,155.7361","-154,453.2037",-1.4724,False,linear


## 8. Approved interpolated-ratio certificate bubble

This section plots the previously calculated production output in `copper_certificate_bubble.csv`. It is kept separate from the experimental regression bubble above.

In [37]:
approved_bubble_path = first_existing(
    "../data/processed/bubble/copper_certificate_bubble.csv",
    "commodity/copper/data/processed/bubble/copper_certificate_bubble.csv",
)
approved_bubble = pd.read_csv(approved_bubble_path, parse_dates=["date"])
approved_bubble["certificate_bubble_pct"] = pd.to_numeric(
    approved_bubble["certificate_bubble_pct"], errors="raise"
)
approved_bubble = approved_bubble.sort_values("date").reset_index(drop=True)
approved_bubble["is_observed_ratio"] = approved_bubble["physical_ratio_method"].eq("observed")
print(
    f'Primary bubble: {len(approved_bubble):,} dates, '
    f'{approved_bubble["is_observed_ratio"].sum():,} exact anchors, '
    f'{(~approved_bubble["is_observed_ratio"]).sum():,} interpolated dates; '
    f'{approved_bubble["date"].min().date()} to {approved_bubble["date"].max().date()}'
)
print(
    f'Mean premium: {approved_bubble["certificate_bubble_pct"].mean():.2f}%; '
    f'median: {approved_bubble["certificate_bubble_pct"].median():.2f}%'
)

observed_ratio_days = approved_bubble.loc[approved_bubble['is_observed_ratio']]
fig = go.Figure()
fig.add_trace(go.Scatter(x=approved_bubble['date'], y=approved_bubble['certificate_bubble_pct'],
    name='Interpolated physical ratio', line_color='#1976D2'))
fig.add_trace(go.Scatter(x=observed_ratio_days['date'], y=observed_ratio_days['certificate_bubble_pct'],
    mode='markers', name='Observed physical ratio', marker=dict(color='#C62828', size=8)))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title='Copper certificate bubble: bounded physical-ratio estimate',
    xaxis_title='Date', yaxis_title='Bubble (%)', height=550,
    template='plotly_white', hovermode='x unified')
fig.show()

Primary bubble: 206 dates, 36 exact anchors, 170 interpolated dates; 2025-10-26 to 2026-09-20
Mean premium: 5.72%; median: 7.10%


## 9. Observed IME physical-market premium over intrinsic copper value

For actual physical-market observations in the regression period, intrinsic value is `LME cash USD/kg × free-market USD/IRR`. The plotted premium is `(observed physical price / intrinsic price - 1) × 100`.

In [38]:
physical_intrinsic_bubble = anchors[[
    "date", "physical_price", "lme_usd_per_kg", "usd_irr", "intrinsic_price"
]].copy()
assert len(physical_intrinsic_bubble) == len(anchors)
physical_intrinsic_bubble["physical_vs_intrinsic_bubble_pct"] = (
    physical_intrinsic_bubble["physical_price"]
    / physical_intrinsic_bubble["intrinsic_price"]
    - 1
) * 100

fig = go.Figure(go.Scatter(x=physical_intrinsic_bubble['date'],
    y=physical_intrinsic_bubble['physical_vs_intrinsic_bubble_pct'], mode='lines+markers',
    name='Observed IME physical premium', line_color='#7B1FA2'))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title='Observed IME physical copper vs intrinsic LME-FX value',
    xaxis_title='Physical trade date', yaxis_title='Premium / discount (%)',
    height=550, template='plotly_white')
fig.show()
physical_intrinsic_bubble

,date,physical_price,lme_usd_per_kg,usd_irr,intrinsic_price,physical_vs_intrinsic_bubble_pct
0,2025-11-23,"10,660,673.0000",10.6855,"1,132,000.0000","12,095,986.0000",-11.8660
1,2025-12-17,"13,266,993.0000",11.7200,"1,310,800.0000","15,362,576.0000",-13.6408
2,2026-03-29,"16,541,000.0000",12.0460,"1,585,600.0000","19,100,137.6000",-13.3985
3,2026-03-30,"16,541,000.0000",12.1370,"1,585,600.0000","19,244,427.2000",-14.0478
4,2026-04-05,"16,541,000.0000",12.1470,"1,527,800.0000","18,558,186.6000",-10.8695
5,2026-04-12,"16,766,000.0000",12.6605,"1,553,100.0000","19,663,022.5500",-14.7334
6,2026-04-13,"16,766,000.0000",12.8205,"1,578,300.0000","20,234,595.1500",-17.1419
7,2026-04-19,"17,604,000.0000",13.1490,"1,507,300.0000","19,819,487.7000",-11.1783
8,2026-04-26,"18,484,000.0000",13.2300,"1,569,500.0000","20,764,485.0000",-10.9826
9,2026-05-03,"19,308,847.0000",12.8950,"1,869,000.0000","24,100,755.0000",-19.8828


## 10. Certificate premium over intrinsic copper value

This measure compares certificate VWAP directly with `LME cash USD/kg × free-market USD/IRR`; it does not use observed, interpolated, or regression-estimated IME physical price.

In [39]:
certificate_intrinsic_bubble = certificate_features[[
    "date", "certificate_price", "lme_usd_per_kg", "usd_irr", "intrinsic_price"
]].copy()
certificate_intrinsic_bubble["certificate_vs_intrinsic_bubble_pct"] = (
    certificate_intrinsic_bubble["certificate_price"]
    / certificate_intrinsic_bubble["intrinsic_price"]
    - 1
) * 100

fig = go.Figure(go.Scatter(x=certificate_intrinsic_bubble['date'],
    y=certificate_intrinsic_bubble['certificate_vs_intrinsic_bubble_pct'],
    name='Certificate premium over intrinsic value', line_color='#EF6C00'))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title='Copper certificate vs intrinsic LME-FX value',
    xaxis_title='Date', yaxis_title='Premium / discount (%)',
    height=550, template='plotly_white')
fig.show()
certificate_intrinsic_bubble

,date,certificate_price,lme_usd_per_kg,usd_irr,intrinsic_price,certificate_vs_intrinsic_bubble_pct
0,2025-10-20,"10,507,161.1641",10.5810,"1,081,500.0000","11,443,351.5000",-8.1811
1,2025-10-21,"10,454,394.0267",10.6120,"1,075,500.0000","11,413,206.0000",-8.4009
2,2025-10-22,"10,215,889.4684",10.6000,"1,076,000.0000","11,405,600.0000",-10.4309
3,2025-10-25,"10,205,657.0950",10.8070,"1,072,000.0000","11,585,104.0000",-11.9071
4,2025-10-26,"10,335,702.5324",10.8070,"1,078,000.0000","11,649,946.0000",-11.2811
...,...,...,...,...,...,...
205,2026-09-14,"28,213,935.4965",14.0440,"2,313,000.0000","32,483,772.0000",-13.1445
206,2026-09-15,"27,457,542.6940",14.0450,"2,303,950.0000","32,358,977.7500",-15.1471
207,2026-09-16,"27,612,045.3288",14.2270,"2,305,000.0000","32,793,235.0000",-15.7996
208,2026-09-19,"27,772,097.7749",14.5290,"2,302,750.0000","33,456,654.7500",-16.9908


## 11. Full-history IME physical copper premium over intrinsic value

This section extends the physical-market analysis to the complete available history. It uses every positive-trade observation in `nci_copper_cash_daily.csv`, not only the recent certificate period. Intrinsic value is `LME cash USD/kg × free-market USD/IRR`.

In [40]:
physical_full_history = add_market_inputs(physical).copy()
physical_full_history["physical_vs_intrinsic_bubble_pct"] = (
    physical_full_history["physical_price"]
    / physical_full_history["intrinsic_price"]
    - 1
) * 100

assert len(physical_full_history) == len(physical)
assert physical_full_history["physical_vs_intrinsic_bubble_pct"].notna().all()

print(f"Physical observations: {len(physical_full_history):,}")
print(
    f"Coverage: {physical_full_history['date'].min().date()} "
    f"through {physical_full_history['date'].max().date()}"
)
physical_full_history[[
    "date",
    "physical_price",
    "lme_usd_per_kg",
    "usd_irr",
    "intrinsic_price",
    "physical_vs_intrinsic_bubble_pct",
]].describe(include="all")

Physical observations: 799
Coverage: 2008-08-24 through 2026-09-20


,date,physical_price,lme_usd_per_kg,usd_irr,intrinsic_price,physical_vs_intrinsic_bubble_pct
count,799,799.0000,799.0000,799.0000,799.0000,799.0000
mean,2017-04-14 12:53:09.987484,"1,894,364.8020",7.4229,"219,305.2203","2,234,528.7396",-8.5752
min,2008-08-24 00:00:00,"28,636.0000",2.7700,"9,670.0000","27,561.5000",-57.5203
25%,2012-12-26 12:00:00,"161,563.0000",5.9620,"29,515.5000","162,974.7375",-15.1578
50%,2017-02-19 00:00:00,"227,959.0000",7.0960,"37,737.5000","253,609.2750",-7.0871
75%,2021-07-29 12:00:00,"2,133,181.5000",8.6010,"261,650.0000","2,379,988.3413",-1.1772
max,2026-09-20 00:00:00,"26,828,473.0000",14.8500,"2,306,000.0000","33,503,874.0000",28.0412
std,NaN,"4,197,198.0634",2.0741,"386,866.9578","5,079,942.6072",10.5543


In [41]:
fig = go.Figure(go.Scatter(x=physical_full_history['date'],
    y=physical_full_history['physical_vs_intrinsic_bubble_pct'], mode='lines+markers',
    marker_size=4, name='Observed IME physical premium', line_color='#7B1FA2'))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title='Full-history IME physical copper vs intrinsic LME-FX value',
    xaxis_title='Physical trade date', yaxis_title='Premium / discount (%)',
    height=600, template='plotly_white')
fig.show()

## 12. Corrected regression: physical price on intrinsic LME–FX value

This section supersedes the earlier experimental two-feature regression. The sole explanatory variable is intrinsic copper value: `LME cash USD/kg × free-market USD/IRR`. Certificate price is not used in model fitting. Three specifications are compared: proportional (no intercept), linear with intercept, and degree-2 polynomial Ridge.

In [42]:
INTRINSIC_FEATURE = ["intrinsic_price"]
X_intrinsic = anchors[INTRINSIC_FEATURE]
y_physical = anchors["physical_price"]

intrinsic_models = {
    "proportional_no_intercept": LinearRegression(fit_intercept=False),
    "linear_with_intercept": LinearRegression(),
    "polynomial_degree_2_ridge": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-4, 4, 81))),
    ]),
}

intrinsic_comparison_rows = []
intrinsic_oos_predictions = {}
intrinsic_tscv = TimeSeriesSplit(n_splits=5)

for name, model in intrinsic_models.items():
    prediction = pd.Series(np.nan, index=y_physical.index, dtype=float)
    for train_index, test_index in intrinsic_tscv.split(X_intrinsic):
        fold_model = clone(model)
        fold_model.fit(X_intrinsic.iloc[train_index], y_physical.iloc[train_index])
        prediction.iloc[test_index] = fold_model.predict(X_intrinsic.iloc[test_index])
    valid = prediction.notna()
    actual = y_physical.loc[valid]
    predicted = prediction.loc[valid]
    intrinsic_oos_predictions[name] = predicted
    intrinsic_comparison_rows.append({
        "model": name,
        "oos_n": len(actual),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted),
    })

intrinsic_model_comparison = (
    pd.DataFrame(intrinsic_comparison_rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
intrinsic_model_comparison

,model,oos_n,MAE,RMSE,R2
0,linear_with_intercept,25,"469,838.7717","662,671.6639",0.8957
1,polynomial_degree_2_ridge,25,"545,425.2128","746,477.9852",0.8677
2,proportional_no_intercept,25,"841,481.7510","1,079,049.3075",0.7235


In [43]:
selected_intrinsic_model_name = intrinsic_model_comparison.loc[0, "model"]
selected_intrinsic_model = clone(intrinsic_models[selected_intrinsic_model_name])
selected_intrinsic_model.fit(X_intrinsic, y_physical)

anchors_intrinsic_regression = anchors.copy()
anchors_intrinsic_regression["fitted_physical_price"] = (
    selected_intrinsic_model.predict(anchors_intrinsic_regression[INTRINSIC_FEATURE])
)
anchors_intrinsic_regression["fitted_error_pct"] = (
    anchors_intrinsic_regression["fitted_physical_price"]
    / anchors_intrinsic_regression["physical_price"]
    - 1
) * 100

bubble_intrinsic_regression = certificate_features.copy()
bubble_intrinsic_regression["estimated_physical_price"] = (
    selected_intrinsic_model.predict(bubble_intrinsic_regression[INTRINSIC_FEATURE])
)
if (bubble_intrinsic_regression["estimated_physical_price"] <= 0).any():
    raise ValueError("The selected intrinsic regression produced a non-positive price")
bubble_intrinsic_regression["certificate_bubble_pct"] = (
    bubble_intrinsic_regression["certificate_price"]
    / bubble_intrinsic_regression["estimated_physical_price"]
    - 1
) * 100
bubble_intrinsic_regression["is_physical_anchor_date"] = (
    bubble_intrinsic_regression["date"].isin(anchors["date"])
)

print(f"Selected corrected model: {selected_intrinsic_model_name}")
anchors_intrinsic_regression[[
    "date", "intrinsic_price", "physical_price",
    "fitted_physical_price", "fitted_error_pct"
]]

Selected corrected model: linear_with_intercept


,date,intrinsic_price,physical_price,fitted_physical_price,fitted_error_pct
0,2025-11-23,"12,095,986.0000","10,660,673.0000","11,797,596.6689",10.6647
1,2025-12-17,"15,362,576.0000","13,266,993.0000","14,021,396.0833",5.6863
2,2026-03-29,"19,100,137.6000","16,541,000.0000","16,565,819.2579",0.1500
3,2026-03-30,"19,244,427.2000","16,541,000.0000","16,664,047.4187",0.7439
4,2026-04-05,"18,558,186.6000","16,541,000.0000","16,196,874.7878",-2.0804
5,2026-04-12,"19,663,022.5500","16,766,000.0000","16,949,014.9519",1.0916
6,2026-04-13,"20,234,595.1500","16,766,000.0000","17,338,124.9453",3.4124
7,2026-04-19,"19,819,487.7000","17,604,000.0000","17,055,531.8729",-3.1156
8,2026-04-26,"20,764,485.0000","18,484,000.0000","17,698,858.5354",-4.2477
9,2026-05-03,"24,100,755.0000","19,308,847.0000","19,970,094.0662",3.4246


In [44]:
fit_limits = [min(anchors_intrinsic_regression['physical_price'].min(),
                  anchors_intrinsic_regression['fitted_physical_price'].min()),
              max(anchors_intrinsic_regression['physical_price'].max(),
                  anchors_intrinsic_regression['fitted_physical_price'].max())]
anchor_regression_bubbles = bubble_intrinsic_regression.loc[
    bubble_intrinsic_regression['is_physical_anchor_date']]
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f'Observed vs fitted: {selected_intrinsic_model_name}', 'Certificate bubble: intrinsic regression'))
fig.add_trace(go.Scatter(x=anchors_intrinsic_regression['physical_price'],
    y=anchors_intrinsic_regression['fitted_physical_price'], mode='markers',
    name='Physical anchors', marker_color='#1976D2'), row=1, col=1)
fig.add_trace(go.Scatter(x=fit_limits, y=fit_limits, name='Perfect fit',
    line=dict(color='#455A64', dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=bubble_intrinsic_regression['date'],
    y=bubble_intrinsic_regression['certificate_bubble_pct'],
    name='Regression bubble', line_color='#7B1FA2'), row=1, col=2)
fig.add_trace(go.Scatter(x=anchor_regression_bubbles['date'],
    y=anchor_regression_bubbles['certificate_bubble_pct'], mode='markers',
    name='Physical anchor date', marker_color='#C62828'), row=1, col=2)
fig.add_hline(y=0, line_dash='dash', line_color='#455A64', row=1, col=2)
fig.update_xaxes(title_text='Observed physical price (IRR/kg)', row=1, col=1)
fig.update_yaxes(title_text='Fitted physical price (IRR/kg)', row=1, col=1)
fig.update_xaxes(title_text='Date', row=1, col=2)
fig.update_yaxes(title_text='Bubble (%)', row=1, col=2)
fig.update_layout(height=550, template='plotly_white')
fig.show()

## 13. Forward trades inside the 102-day cash gap

The gap is a gap in the exact NCI copper-cathode cash benchmark, not a market closure. This section isolates exact-symbol forward trades strictly inside the gap and compares their weighted prices with the last cash price before the gap, the first cash price after it, and a straight-line bridge between those two cash anchors.

In [45]:
forward_gap_path = first_existing(
    "../data/processed/physical/nci_copper_forward_gap.csv",
    "commodity/copper/data/processed/physical/nci_copper_forward_gap.csv",
)
forward_gap = pd.read_csv(forward_gap_path)
forward_gap["date"] = pd.to_datetime(forward_gap["trade_date_gregorian"])

forward_gap_summary = pd.Series(
    {
        "forward trade dates": len(forward_gap),
        "total quantity (t)": forward_gap["total_quantity"].sum(),
        "quantity-weighted price (IRR/kg)": np.average(
            forward_gap["forward_weighted_price"],
            weights=forward_gap["total_quantity"],
        ),
        "previous cash price (IRR/kg)": forward_gap["previous_cash_price"].iloc[0],
        "next cash price (IRR/kg)": forward_gap["next_cash_price"].iloc[0],
    },
    name="value",
)
display(forward_gap_summary.to_frame())
display(
    forward_gap[
        [
            "trade_date_jalali",
            "contract_types",
            "total_quantity",
            "forward_weighted_price",
            "vs_previous_cash_pct",
            "vs_next_cash_pct",
            "vs_linear_bridge_pct",
        ]
    ].round(2)
)

,value
forward trade dates,16.0000
total quantity (t),"26,420.0000"
quantity-weighted price (IRR/kg),"16,327,124.9054"
previous cash price (IRR/kg),"13,266,993.0000"
next cash price (IRR/kg),"16,541,000.0000"


,trade_date_jalali,contract_types,total_quantity,forward_weighted_price,vs_previous_cash_pct,vs_next_cash_pct,vs_linear_bridge_pct
0,1404/10/01,سلف,3000,14216860,7.1600,-14.0500,5.8800
1,1404/10/07,سلف,3000,16077020,21.1800,-2.8100,18.0400
2,1404/10/15,سلف|سلف (مچینگ),3000,15834000,19.3500,-4.2700,14.1000
3,1404/10/21,سلف|سلف (مچینگ),3000,15834000,19.3500,-4.2700,12.5400
4,1404/10/29,سلف|سلف (مچینگ),2520,16626000,25.3200,0.5100,16.0500
5,1404/11/05,سلف|سلف (مچینگ),2360,16340000,23.1600,-1.2200,12.5400
6,1404/11/12,سلف|سلف (مچینگ),3000,16892000,27.3200,2.1200,14.5700
7,1404/11/19,سلف|سلف (مچینگ),2200,17601000,32.6700,6.4100,17.5900
8,1404/11/20,سلف (مچینگ),180,17601000,32.6700,6.4100,17.3400
9,1404/11/21,سلف (مچینگ),20,17601000,32.6700,6.4100,17.0900


In [46]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
    subplot_titles=('Forward prices inside the 102-day cash gap', 'Difference from cash anchors'))
dates = forward_gap['trade_date_jalali']
for column, name, color in [
    ('forward_weighted_price', 'Forward weighted price', '#1976D2'),
    ('linear_cash_bridge_price', 'Linear cash bridge', '#EF6C00'),
]:
    fig.add_trace(go.Scatter(x=dates, y=forward_gap[column], name=name,
        mode='lines+markers', line_color=color), row=1, col=1)
for column, name, color in [
    ('previous_cash_price', 'Previous cash anchor', '#546E7A'),
    ('next_cash_price', 'Next cash anchor', '#2E7D32'),
]:
    fig.add_hline(y=forward_gap[column].iloc[0], line_color=color,
        annotation_text=name, row=1, col=1)
for column, name, color in [
    ('vs_previous_cash_pct', 'vs previous cash', '#1976D2'),
    ('vs_next_cash_pct', 'vs next cash', '#00897B'),
    ('vs_linear_bridge_pct', 'vs linear bridge', '#EF6C00'),
]:
    fig.add_trace(go.Bar(x=dates, y=forward_gap[column], name=name,
        marker_color=color), row=2, col=1)
fig.add_hline(y=0, line_color='#455A64', row=2, col=1)
fig.update_yaxes(title_text='IRR/kg', row=1, col=1)
fig.update_yaxes(title_text='Difference (%)', row=2, col=1)
fig.update_xaxes(title_text='Trade date (Jalali)', row=2, col=1)
fig.update_layout(height=850, barmode='group', template='plotly_white')
fig.show()

**Interpretation.** The 16 forward-trade dates total 26,420 tonnes. Forward prices range from 7.16% to 33.73% above the previous cash anchor and from 14.05% below to 7.26% above the next cash anchor. These observations are forward contracts with cash/credit settlement terms; they should not be inserted into the primary cash-only benchmark without explicit maturity and financing adjustments.

## 14. Certificate trading volume through time

The table and chart use every positive-volume copper-cathode certificate trading day in the canonical certificate input. Volume is the number of traded warehouse receipts reported by IME.

In [47]:
certificate_volume_history = (
    certificate[["date", "TradesVolume", "TradesValue"]]
    .rename(
        columns={
            "TradesVolume": "certificate_trades_volume",
            "TradesValue": "certificate_trades_value_irr",
        }
    )
    .sort_values("date")
    .reset_index(drop=True)
)
assert certificate_volume_history["certificate_trades_volume"].gt(0).all()
display(certificate_volume_history)

,date,certificate_trades_volume,certificate_trades_value_irr
0,2025-10-20,57651,605748348270
1,2025-10-21,26322,275180559570
2,2025-10-22,90627,925835414850
3,2025-10-25,13043,133112385490
4,2025-10-26,70918,732987352190
...,...,...,...
205,2026-09-14,93511,2638313322210
206,2026-09-15,48809,1340175201350
207,2026-09-16,43312,1195932907280
208,2026-09-19,60419,1677962375460


In [48]:
fig = go.Figure(go.Bar(x=certificate_volume_history['date'],
    y=certificate_volume_history['certificate_trades_volume'], marker_color='#1976D2'))
fig.update_layout(title='Copper-cathode certificate trading volume',
    xaxis_title='Trade date', yaxis_title='Traded certificates',
    height=550, template='plotly_white')
fig.show()

## 15. Certificate bubble on jointly observed dates only

This diagnostic removes all interpolated dates. Bubble is recalculated only when certificate and comparable NCI cash physical prices are both observed on the same day: `(certificate price / observed physical price - 1) × 100`.

In [49]:
observed_certificate_bubble = approved_bubble.loc[
    approved_bubble["physical_ratio_method"].eq("observed"),
    [
        "date",
        "certificate_trades_volume",
        "certificate_price_irr_per_kg",
        "observed_physical_price_irr_per_kg",
    ],
].copy()
observed_certificate_bubble["observed_certificate_bubble_irr_per_kg"] = (
    observed_certificate_bubble["certificate_price_irr_per_kg"]
    - observed_certificate_bubble["observed_physical_price_irr_per_kg"]
)
observed_certificate_bubble["observed_certificate_bubble_pct"] = (
    observed_certificate_bubble["certificate_price_irr_per_kg"]
    / observed_certificate_bubble["observed_physical_price_irr_per_kg"]
    - 1
) * 100
observed_certificate_bubble = observed_certificate_bubble.sort_values("date").reset_index(drop=True)
assert len(observed_certificate_bubble) == approved_bubble["physical_ratio_method"].eq("observed").sum()
display(observed_certificate_bubble.round(2))

C:\Users\amirabadi\AppData\Local\Temp\ipykernel_9852\656558480.py:21: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(observed_certificate_bubble.round(2))


,date,certificate_trades_volume,certificate_price_irr_per_kg,observed_physical_price_irr_per_kg,observed_certificate_bubble_irr_per_kg,observed_certificate_bubble_pct
0,2025-10-26,70918,10335703,"10,296,373.0000","39,330.0000",0.3800
1,2025-11-23,10106,10693107,"10,660,673.0000","32,434.0000",0.3000
2,2025-12-17,84189,14160408,"13,266,993.0000","893,415.0000",6.7300
3,2026-03-29,838,16132371,"16,541,000.0000","-408,629.0000",-2.4700
4,2026-03-30,2816,16144178,"16,541,000.0000","-396,822.0000",-2.4000
5,2026-04-05,4547,15590354,"16,541,000.0000","-950,646.0000",-5.7500
6,2026-04-12,5364,16114179,"16,766,000.0000","-651,821.0000",-3.8900
7,2026-04-13,9273,16425974,"16,766,000.0000","-340,026.0000",-2.0300
8,2026-04-19,4110,16130906,"17,604,000.0000","-1,473,094.0000",-8.3700
9,2026-04-26,30123,18010209,"18,484,000.0000","-473,791.0000",-2.5600


In [50]:
colors = np.where(observed_certificate_bubble['observed_certificate_bubble_pct'].ge(0),
    '#C62828', '#2E7D32')
fig = go.Figure(go.Bar(x=observed_certificate_bubble['date'],
    y=observed_certificate_bubble['observed_certificate_bubble_pct'],
    marker_color=colors, name='Observed exact-date bubble'))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title=f'Copper certificate bubble: {len(observed_certificate_bubble)} exact-date anchors',
    xaxis_title='Trade date', yaxis_title='Observed bubble (%)',
    height=550, template='plotly_white')
fig.show()

## Standard market dashboard

This governed, read-only section uses the same presentation contract across commodity projects:
source coverage, physical and certificate activity, separate price panels, physical-goods
composition, and validated processed bubbles. It never writes raw data or constructs a missing
bubble. For Copper, product comparability still follows the project-specific workflow.

In [51]:
from pathlib import Path
import sys

def locate_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "commodity" / "copper").exists() and (candidate / "shared").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

WORKSPACE_ROOT = locate_workspace()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from shared.notebook_tools.commodity_dashboard import (
    goods_type_counts,
    load_markets,
    market_summary,
    plot_available_bubbles,
    plot_goods_type_counts,
    plot_market_prices,
    plot_trade_activity,
)

PROJECT_DIR = WORKSPACE_ROOT / "commodity" / "copper"
physical_dashboard, certificate_dashboard = load_markets(
    PROJECT_DIR, "copper", physical_filename='copper_cathode_physical_raw.csv'
)
display(market_summary(physical_dashboard, certificate_dashboard))
plot_trade_activity(physical_dashboard, certificate_dashboard, "Copper")
plot_market_prices(physical_dashboard, certificate_dashboard, "Copper")
goods_count_table = plot_goods_type_counts(physical_dashboard, "Copper", top_n=30)
display(goods_count_table)
bubble_series_plotted = plot_available_bubbles(PROJECT_DIR, "Copper")

,source_records,positive_trade_records,trading_days,first_date,last_date
market,,,,,
physical,1175,1166,799,1387/06/03,1405/06/29
certificate,288,210,210,1404/07/28,1405/06/29


Distinct physical GoodsName labels: 1; records counted: 1,175


,goods_name,record_count,share_pct
0,مس کاتد,1175,100.0000


## Historical bubble distribution

This section reads the standardized processed table and renders an interactive Plotly figure for
each bubble type. The panels show the observed distribution, empirical cumulative distribution
function F(x), and magnitude frequency P(|Bubble| >= |x|). Negative bubbles retain their sign in
the first two panels; the third panel measures magnitude only.

In [52]:
from pathlib import Path
import sys
import pandas as pd

def locate_distribution_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "shared").exists() and (candidate / "commodity/copper").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

distribution_workspace = locate_distribution_workspace()
if str(distribution_workspace) not in sys.path:
    sys.path.insert(0, str(distribution_workspace))

from shared.market_analysis.bubble_distribution import plot_distribution_plotly

distribution_project = distribution_workspace / "commodity/copper"
distribution_files = list(
    (distribution_project / "data/processed/bubble").glob("*_bubble_distribution.csv")
)
if len(distribution_files) != 1:
    raise ValueError(f"Expected one named bubble distribution CSV, found {distribution_files}")
bubble_distribution = pd.read_csv(distribution_files[0], parse_dates=["observation_date"])
for series_id, series_distribution in bubble_distribution.groupby("series_id", sort=True):
    comparison = series_distribution["comparison"].iloc[0]
    figure = plot_distribution_plotly(series_distribution, comparison)
    figure.show()

display(bubble_distribution)

,commodity,bubble_type,point_type,point_method,is_interpolated,series_id,comparison,source_file,observation_date,bubble_pct,empirical_cdf,percentile,abs_exceedance_probability,abs_exceedance_pct,observation_count
0,copper,certificate_vs_intrinsic,observed,observed,False,certificate_intrinsic,Certificate vs intrinsic,certificate_vs_intrinsic_bubble.csv,2026-03-16,-26.0036,0.0048,0.4762,0.0048,0.4762,210
1,copper,certificate_vs_intrinsic,observed,observed,False,certificate_intrinsic,Certificate vs intrinsic,certificate_vs_intrinsic_bubble.csv,2026-02-25,-24.4009,0.0095,0.9524,0.0095,0.9524,210
2,copper,certificate_vs_intrinsic,observed,observed,False,certificate_intrinsic,Certificate vs intrinsic,certificate_vs_intrinsic_bubble.csv,2026-02-22,-22.6585,0.0143,1.4286,0.0143,1.4286,210
3,copper,certificate_vs_intrinsic,observed,observed,False,certificate_intrinsic,Certificate vs intrinsic,certificate_vs_intrinsic_bubble.csv,2026-09-02,-22.4550,0.0190,1.9048,0.0190,1.9048,210
4,copper,certificate_vs_intrinsic,observed,observed,False,certificate_intrinsic,Certificate vs intrinsic,certificate_vs_intrinsic_bubble.csv,2026-02-24,-22.1083,0.0238,2.3810,0.0238,2.3810,210
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,copper,physical_vs_intrinsic,observed,observed,False,physical_intrinsic,Physical vs intrinsic,physical_vs_intrinsic_bubble.csv,2008-10-26,15.7529,0.9950,99.4994,0.2428,24.2804,799
1211,copper,physical_vs_intrinsic,observed,observed,False,physical_intrinsic,Physical vs intrinsic,physical_vs_intrinsic_bubble.csv,2009-02-11,17.5504,0.9962,99.6245,0.1765,17.6471,799
1212,copper,physical_vs_intrinsic,observed,observed,False,physical_intrinsic,Physical vs intrinsic,physical_vs_intrinsic_bubble.csv,2009-02-28,19.3065,0.9975,99.7497,0.1452,14.5181,799
1213,copper,physical_vs_intrinsic,observed,observed,False,physical_intrinsic,Physical vs intrinsic,physical_vs_intrinsic_bubble.csv,2009-01-25,26.7244,0.9987,99.8748,0.0538,5.3817,799
